In [1]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 5.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554980 sha256=7bd59395dc700f944854bda99a828c963421535e52cf417928115fdf6b3275ab
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [4]:
import pandas as pd

df = pd.read_csv("coursea_data.csv")
df.head()

,Unnamed: 0,course_title,course_organization,course_Certificate_type,course_rating,course_difficulty,course_students_enrolled
0,134,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k
1,743,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k
2,874,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k
3,413,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k
4,635,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k


In [5]:
# Remove unnecessary column
df = df.drop(columns=['Unnamed: 0'])

# Check for missing values
print(df.isnull().sum())

# Convert rating column to float
df['course_rating'] = df['course_rating'].astype(float)

df.head()


course_title                0
course_organization         0
course_Certificate_type     0
course_rating               0
course_difficulty           0
course_students_enrolled    0
dtype: int64


,course_title,course_organization,course_Certificate_type,course_rating,course_difficulty,course_students_enrolled
0,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k
1,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k
2,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k
3,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k
4,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k


In [6]:
df['features'] = df['course_title'] + " " + \
                 df['course_organization'] + " " + \
                 df['course_difficulty'] + " " + \
                 df['course_Certificate_type']

df[['course_title','features']].head()

,course_title,features
0,(ISC)² Systems Security Certified Practitioner...,(ISC)² Systems Security Certified Practitioner...
1,A Crash Course in Causality: Inferring Causal...,A Crash Course in Causality: Inferring Causal...
2,A Crash Course in Data Science,A Crash Course in Data Science Johns Hopkins U...
3,A Law Student's Toolkit,A Law Student's Toolkit Yale University Mixed ...
4,A Life of Happiness and Fulfillment,A Life of Happiness and Fulfillment Indian Sch...


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words='english')

feature_matrix = vectorizer.fit_transform(df['features'])

feature_matrix.shape

(891, 1551)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)

similarity_matrix

array([[1.        , 0.        , 0.        , ..., 0.0139851 , 0.01490684,
        0.01867086],
       [0.        , 1.        , 0.31466568, ..., 0.02085652, 0.02954353,
        0.00577912],
       [0.        , 0.31466568, 1.        , ..., 0.        , 0.01190043,
        0.00940514],
       ...,
       [0.0139851 , 0.02085652, 0.        , ..., 1.        , 0.0416849 ,
        0.01360031],
       [0.01490684, 0.02954353, 0.01190043, ..., 0.0416849 , 1.        ,
        0.02127881],
       [0.01867086, 0.00577912, 0.00940514, ..., 0.01360031, 0.02127881,
        1.        ]])

In [10]:
def recommend_courses(course_name):

    idx = df[df['course_title'] == course_name].index[0]

    similarity_scores = list(enumerate(similarity_matrix[idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    top_courses = similarity_scores[1:6]

    for i in top_courses:
        print(df.iloc[i[0]]['course_title'])

In [11]:
recommend_courses("Machine Learning")

Machine Learning for All
Machine Learning with Python
Machine Learning Foundations: A Case Study Approach
Machine Learning
Machine Learning and Reinforcement Learning in Finance


In [12]:
course_to_recommend = "Database Management System"
if course_to_recommend in df['course_title'].values:
    recommend_courses(course_to_recommend)
else:
    print(f"Course '{course_to_recommend}' not found in the dataset. Please provide an existing course title.")

IndexError: index 0 is out of bounds for axis 0 with size 0

In [13]:
recommend_courses("Python for Everybody")

Programming for Everybody (Getting Started with Python)
Web Applications for Everybody
Statistics with Python
Python Basics
Introduction to Data Science in Python


In [14]:
recommend_courses("Business Foundations")

Business Analytics
Foundations of Positive Psychology
Foundations of Business Strategy
Foundations of Management
The Business of Health Care


In [15]:
def recommend_courses(course_name):

    course_name = course_name.lower()

    matches = df[df['course_title'].str.lower().str.contains(course_name)]

    if matches.empty:
        print("Course not found in dataset")
        return

    idx = matches.index[0]

    similarity_scores = list(enumerate(similarity_matrix[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    top_courses = similarity_scores[1:6]

    for i in top_courses:
        print(df.iloc[i[0]]['course_title'])

In [16]:
recommend_courses("database")

What is Data Science?
SQL for Data Science
Introduction to Data Science
IBM Data Science
Tools for Data Science


In [17]:
recommend_courses("spiritual health")

Course not found in dataset


In [18]:
def recommend_courses(course_name, num_recommendations=5):

    course_name = course_name.lower()

    matches = df[df['course_title'].str.lower().str.contains(course_name)]

    if matches.empty:
        print("Course not found in dataset")
        return

    idx = matches.index[0]

    similarity_scores = list(enumerate(similarity_matrix[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    top_courses = similarity_scores[1:num_recommendations+1]

    recommended_indices = [i[0] for i in top_courses]

    result = df.iloc[recommended_indices][[
        'course_title',
        'course_organization',
        'course_rating',
        'course_difficulty'
    ]]

    result = result.reset_index(drop=True)
    result.index = result.index + 1

    return result

In [19]:
recommend_courses("machine")

,course_title,course_organization,course_rating,course_difficulty
1,Mathematics for Data Science,National Research University Higher School of ...,4.5,Beginner
2,Основы Digital Маркетинга,National Research University Higher School of ...,4.5,Intermediate
3,Introduction to Discrete Mathematics for Compu...,National Research University Higher School of ...,4.4,Beginner
4,Основы программирования на Python,National Research University Higher School of ...,4.6,Beginner
5,Корпоративные финансы и стоимость компании,National Research University Higher School of ...,4.5,Beginner


In [20]:
recommend_courses("operating system")

,course_title,course_organization,course_rating,course_difficulty
1,Using Python to Interact with the Operating Sy...,Google,4.7,Beginner
2,User Experience Research and Design,University of Michigan,4.8,Beginner
3,Microeconomics: The Power of Markets,University of Pennsylvania,4.7,Mixed
4,Introduction to User Experience Design,Georgia Institute of Technology,4.5,Beginner
5,Introduction to User Experience Principles and...,University of Michigan,4.8,Beginner


In [22]:
!git clone https://github.com/Jolsina/course_guide

Cloning into 'course_guide'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 18 (delta 4), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 62.49 KiB | 1.89 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [25]:
!mv course_recommender.ipynb course_guide/

mv: cannot stat 'course_recommender.ipynb': No such file or directory
